In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

# Load dataset with slightly more features
max_features = 15000  # Vocabulary size
maxlen = 200          # Maximum review length
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)

x_train = pad_sequences(x_train, maxlen=maxlen)
x_test = pad_sequences(x_test, maxlen=maxlen)

# Improved model architecture
model = models.Sequential([
    layers.Embedding(max_features, 128),
    layers.Bidirectional(layers.LSTM(64, dropout=0.2, recurrent_dropout=0.2, return_sequences=True)),
    layers.GlobalMaxPooling1D(),  # Captures the most important features
    layers.Dense(32, activation='relu'),  # Small dense layer
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Early stopping to prevent overfitting and save time
early_stopping = EarlyStopping(
    monitor='val_accuracy',
    patience=2,
    restore_best_weights=True,
    mode='max'
)

# Train with early stopping and larger batch size
model.fit(
    x_train, y_train,
    epochs=10,  # More epochs, but early stopping will prevent unnecessary training
    batch_size=128,  # Increased batch size for faster training
    validation_data=(x_test, y_test),
    callbacks=[early_stopping]
)

test_loss, test_acc = model.evaluate(x_test, y_test)
print(f"Test accuracy: {test_acc}")

print("\nTesting with new reviews...")

# Get the word index from IMDB dataset
word_index = imdb.get_word_index()

# Special tokens handling
word_index = {k:(v+3) for k,v in word_index.items()}
word_index["<PAD>"] = 0
word_index["<START>"] = 1
word_index["<UNK>"] = 2
word_index["<UNUSED>"] = 3

def predict_sentiment(text):
    # Tokenize and convert to sequence
    words = text.lower().split()
    sequence = [word_index.get(word, 2) for word in words]  # 2 = <UNK>
    sequence = [1] + sequence  # Add <START> token
    
    # Pad sequence using the same function as in training
    padded_sequence = pad_sequences([sequence], maxlen=maxlen)
    
    # Make prediction
    prediction = model.predict(padded_sequence, verbose=0)[0][0]
    
    return {
        "text": text,
        "sentiment": "Positive" if prediction > 0.5 else "Negative",
        "probability": float(prediction)
    }

# More diverse test examples
test_reviews = [
    "This movie was amazing, I loved it!",
    "This movie was so boring and terrible.",
    "The film had great special effects but the plot was weak.",
    "I've never been so entertained by such a brilliant screenplay.",
    "The acting was decent, but the story made no sense at all.",
    "This might be the best film I've seen all year.",
    "I couldn't even finish watching, it was that bad.",
    "Beautifully shot with excellent performances from the entire cast."
]

for review in test_reviews:
    result = predict_sentiment(review)
    print(f"Review: {result['text']}")
    print(f"Sentiment: {result['sentiment']} (Probability: {result['probability']:.4f})")
    print()